# MLOps avancé - Validation Evidently AI et Monitoring

Ce notebook approfondit:
- Génération de rapports Evidently AI
- Détection de drift
- Tracking d'expériences avec MLflow
- Visualisations et analyse

## 1. Imports et Configuration

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import logging
from pathlib import Path

sys.path.insert(0, os.getcwd())

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Configuration matplotlib
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ Configuration complète")

## 2. Charger les Données

In [ ]:
from utils.data_loader import load_data, split_data, load_config

# Charger la configuration
config = load_config("utils/config.yaml")
print(f"Configuration chargée: {config}")

# Déterminer le fichier à utiliser
raw_path = "data/raw/"
csv_files = [f for f in os.listdir(raw_path) if f.endswith('.csv')]

if csv_files:
    data_file = os.path.join(raw_path, csv_files[0])
    print(f"\nChargement: {csv_files[0]}")
    
    data = load_data(data_file)
    print(f"\n✓ Données chargées: {data.shape}")
    print(f"\nAperçu:")
    print(data.head())
else:
    print("✗ Aucun fichier CSV trouvé")
    data = None

## 3. Validation des Données avec Evidently AI

In [ ]:
from utils.data_validator import validate_data_quality, create_data_validation_report

if data is not None:
    print("=== VALIDATION AVEC EVIDENTLY ===")
    
    # Validation basique
    print("\n1️⃣ Validation basique...")
    validation = validate_data_quality(data, missing_threshold=0.2, duplicate_threshold=0.1)
    
    print(f"\n  Valeur manquantes: {validation['missing_values_ratio']:.2%}")
    print(f"  Lignes dupliquées: {validation['duplicate_rows_ratio']:.2%}")
    print(f"  Statut global: {'✓ OK' if validation['is_valid'] else '✗ Problèmes détectés'}")
    
    # Rapport Evidently
    print("\n2️⃣ Génération du rapport Evidently...")
    output_dir = "outputs/"
    Path(output_dir).mkdir(exist_ok=True)
    
    report = create_data_validation_report(
        current_data=data,
        output_path=os.path.join(output_dir, "evidently_validation.html")
    )
    
    if report:
        print(f"  ✓ Rapport créé: {os.path.join(output_dir, 'evidently_validation.html')}")
    else:
        print("  ℹ Rapport Evidently Fallback généré")
else:
    print("Aucune donnée")

## 4. Préparation et Entraînement

In [ ]:
from utils.model import train_model, evaluate_model, get_feature_importance

if data is not None:
    print("=== ENTRAÎNEMENT DU MODELE ===")
    
    # Préparer les données
    print("\n1️⃣ Préparation des données...")
    X_train, X_test, y_train, y_test = split_data(data)
    
    # Entraîner
    print("\n2️⃣ Entraînement...")
    model = train_model(X_train, y_train)
    
    if model:
        # Évaluer
        print("\n3️⃣ Évaluation...")
        metrics = evaluate_model(model, X_test, y_test)
        
        print(f"\n📊 Métriques:")
        if metrics:
            for key, value in metrics.items():
                print(f"  {key}: {value:.4f}")
        
        # Feature importance
        print("\n4️⃣ Importance des features...")
        importance = get_feature_importance(model, feature_names=list(X_train.columns))
        if importance:
            print("  Top 10:")
            for feat, imp in list(importance.items())[:10]:
                print(f"    {feat}: {imp:.4f}")

## 5. Détection de Drift

In [ ]:
from utils.performance_monitor import detect_prediction_drift, monitor_model_performance

if model and metrics:
    print("=== MONITORING ET DRIFT DETECTION ===")
    
    # Prédictions
    print("\n1️⃣ Prédictions...")
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)
    
    # Détection de drift
    print("\n2️⃣ Détection de drift...")
    drift_results = detect_prediction_drift(pred_train, pred_test, threshold=0.1)
    
    if drift_results:
        print(f"\n  Drift détecté: {drift_results['drift_detected']}")
        print(f"  Mean drift: {drift_results['mean_drift']:.4f}")
        print(f"  Std drift: {drift_results['std_drift']:.4f}")
        print(f"  Moyenne (ref/current): {drift_results['ref_mean']:.4f} → {drift_results['current_mean']:.4f}")
    
    # Performance monitoring
    print("\n3️⃣ Performance monitoring...")
    perf_metrics = monitor_model_performance(pred_test, y_test)
    if perf_metrics:
        print("  Métriques calculées avec succès")

## 6. MLflow Tracking (Optionnel)

In [ ]:
try:
    from utils.mlflow_tracker import log_training_session
    
    if model and metrics:
        print("=== MLFLOW TRACKING ===")
        print("\n⚠️  MLflow nécessite un serveur en fonctionnement")
        print("   Commande: mlflow ui")
        print("   URL: http://localhost:5000")
        
        # Essayer de logger (peut échouer si MLflow n'est pas disponible)
        try:
            log_training_session(
                model=model,
                metrics=metrics,
                params={
                    'test_size': config['model']['test_size'],
                    'random_state': config['model']['random_state']
                },
                experiment_name=config['mlflow']['experiment_name']
            )
            print("\n✓ Run MLflow loggée avec succès")
        except Exception as e:
            print(f"\n⚠️  MLflow non disponible: {e}")
except ImportError:
    print("⚠️  MLflow non configuré")

## 7. Visualisations

In [ ]:
if model and metrics and importance:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Analyse du Modèle MLOps', fontsize=16, fontweight='bold')
    
    # 1. Feature Importance
    ax = axes[0, 0]
    top_features = dict(list(importance.items())[:10])
    ax.barh(list(top_features.keys()), list(top_features.values()))
    ax.set_xlabel('Importance')
    ax.set_title('Top 10 Features Importance')
    ax.invert_yaxis()
    
    # 2. Métriques
    ax = axes[0, 1]
    metric_names = list(metrics.keys())[:4]
    metric_values = [metrics[k] for k in metric_names]
    ax.bar(metric_names, metric_values)
    ax.set_ylabel('Score')
    ax.set_title('Métriques d\'Évaluation')
    ax.set_ylim(0, 1)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
    
    # 3. Distribution prédictions
    ax = axes[1, 0]
    ax.hist(pred_train, bins=30, alpha=0.6, label='Train', color='blue')
    ax.hist(pred_test, bins=30, alpha=0.6, label='Test', color='orange')
    ax.set_xlabel('Prédictions')
    ax.set_ylabel('Fréquence')
    ax.set_title('Distribution des Prédictions')
    ax.legend()
    
    # 4. Drift info
    ax = axes[1, 1]
    if drift_results:
        ax.text(0.5, 0.8, f"Drift Detected: {drift_results['drift_detected']}",
                ha='center', va='top', fontsize=12, fontweight='bold',
                transform=ax.transAxes)
        ax.text(0.5, 0.6, f"Mean Drift: {drift_results['mean_drift']:.4f}",
                ha='center', va='top', fontsize=10, transform=ax.transAxes)
        ax.text(0.5, 0.4, f"Std Drift: {drift_results['std_drift']:.4f}",
                ha='center', va='top', fontsize=10, transform=ax.transAxes)
    ax.axis('off')
    ax.set_title('Drift Detection Summary')
    
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Visualisations générées")

## 8. Résumé et Conclusions

In [ ]:
print("\n" + "="*60)
print(" "*15 + "RÉSUMÉ PIPELINE MLOps AVANCÉ")
print("="*60)

print("""
✓ Étapes complétées:
  1. ✓ Chargement et diagnostic des données
  2. ✓ Validation avec Evidently AI
  3. ✓ Nettoyage et préparation
  4. ✓ Entraînement du modèle
  5. ✓ Évaluation et métriques
  6. ✓ Analyse de l'importance des features
  7. ✓ Détection de drift
  8. ✓ MLflow tracking (optionnel)
  9. ✓ Visualisations

📊 Résultats:
""")

if metrics:
    print("  Métriques du modèle:")
    for key, value in metrics.items():
        print(f"    {key}: {value:.4f}")

if drift_results:
    print(f"\n  Drift: {drift_results['drift_detected']}")
    print(f"    Mean drift: {drift_results['mean_drift']:.4f}")

print("\n📁 Fichiers générés:")
print(f"  - Rapport Evidently: outputs/evidently_validation.html")
print(f"  - Données traitées: data/processed/")

print("\n🔗 Ressources:")
print("  - Dashboard MLflow: http://localhost:5000")
print("  - Documentation: README_UTILS.md")

print("\n" + "="*60)
print(" "*20 + "✓ PIPELINE OPÉRATIONNEL")
print("="*60)